In [13]:
import pandas as pd
import numpy as np
from faker import Faker
import random

fake = Faker()
np.random.seed(42)
random.seed(42)

# -------------------
# 1. GEO MASTER DATA
# -------------------
regions = []
region_id = 1

geo_data = {
    "Cairo": {"lat": 30.0444, "lon": 31.2357, "multiplier": 1.1},
    "Alexandria": {"lat": 31.2001, "lon": 29.9187, "multiplier": 0.95},
    "Marsa Matruh": {"lat": 31.3543, "lon": 27.2373, "multiplier": 0.9},
    "Port Said": {"lat": 31.2653, "lon": 32.3019, "multiplier": 1.05},
    "Sharm El Sheikh": {"lat": 27.9158, "lon": 34.3299, "multiplier": 1.2},
    "Aswan": {"lat": 24.0889, "lon": 32.8998, "multiplier": 0.85}
}

for city, v in geo_data.items():
    regions.append([
        region_id,
        city,
        v["lat"],
        v["lon"],
        v["multiplier"]
    ])
    region_id += 1

regions_df = pd.DataFrame(regions, columns=[
    "region_id", "region_name", "lat", "lon", "multiplier"
])


# -------------------
# 2. CUSTOMERS
# -------------------

customers = []
industries = ["Retail", "Tech", "Healthcare"]

for i in range(200):  # or your desired number
    # Pick a random region from regions_df
    region_row = regions_df.sample(1).iloc[0]
    
    customers.append([
        i + 1,
        fake.name(),
        region_row["region_id"],  # region_id instead of name
        random.choice(industries)
    ])

customers_df = pd.DataFrame(customers, columns=[
    "customer_id", "customer_name", "region_id", "industry"
])


# -------------------
# 3. PRODUCTS (WITH COST)
# -------------------
product_catalog = {
    "Electronics": ["Laptop", "Smartphone", "Tablet", "Monitor"],
    "Furniture": ["Office Chair", "Desk", "Bookshelf"],
    "Office Supplies": ["Notebook", "Pen Pack", "Stapler"]
}

products = []
product_id = 1

for category, items in product_catalog.items():
    for item in items:
        price = random.randint(100, 1000)
        cost = price * random.uniform(0.5, 0.8)  # cost is 50–80% of price
        products.append([
            product_id,
            item,
            category,
            round(price, 2),
            round(cost, 2)
        ])
        product_id += 1

products_df = pd.DataFrame(products, columns=[
    "product_id", "product_name", "category", "price", "cost"
])

# -------------------
# 4. EMPLOYEES
# -------------------

employees = []

for i in range(50):  # or your desired number
    # Pick a random region from regions_df
    region_row = regions_df.sample(1).iloc[0]
    
    employees.append([
        i + 1,
        fake.name(),
        "Sales",                  # department
        region_row["region_id"]   # region_id
    ])

employees_df = pd.DataFrame(employees, columns=[
    "employee_id", "employee_name", "department", "region_id"
])

# -------------------
# 5. SALES FACT TABLE
# -------------------
sales = []

for i in range(2000):
    product = products_df.sample(1).iloc[0]
    customer_row = customers_df.sample(1).iloc[0]
    employee_row = employees_df[employees_df['region_id'] == customer_row['region_id']].sample(1).iloc[0]
    
    quantity = random.randint(1, 10)
    
    sales.append([
        i + 1,
        fake.date_between(start_date='-1y', end_date='today'),
        product["product_id"],
        customer_row["customer_id"],
        employee_row["employee_id"],
        quantity,
        random.choice([0, 5, 10, 15, 20])  # discount
    ])

sales_df = pd.DataFrame(sales, columns=[
    "sale_id", "date", "product_id", "customer_id", "employee_id", "quantity", "discount"
])

# -------------------
# 6. DATA VALIDATION ✅
# -------------------
assert sales_df["product_id"].isin(products_df["product_id"]).all()
assert sales_df["customer_id"].isin(customers_df["customer_id"]).all()
assert sales_df["employee_id"].isin(employees_df["employee_id"]).all()

# -------------------
# 7. ENRICHMENT
# -------------------

# Merge customer region
sales_df = sales_df.merge(
    customers_df[["customer_id", "region_id"]],
    on="customer_id",
    how="left"
)

sales_df = sales_df.merge(
    regions_df[["region_id", "multiplier"]],
    on="region_id",
    how="left"
)

# Map prices
sales_df["price"] = sales_df["product_id"].map(
    products_df.set_index("product_id")["price"]
)

sales_df["cost"] = sales_df["product_id"].map(
    products_df.set_index("product_id")["cost"]
)

# Adjusted price
sales_df["adjusted_price"] = sales_df["price"] * sales_df["multiplier"]

# -------------------
# 8. DISCOUNT + SEASONALITY
# -------------------
sales_df["discount"] = np.random.choice([0, 5, 10, 15, 20], size=len(sales_df))

sales_df["date"] = pd.to_datetime(sales_df["date"])
sales_df["month"] = sales_df["date"].dt.month

def seasonality(m):
    if m in [11, 12]:
        return 1.3
    elif m in [6, 7]:
        return 0.8
    return 1.0

sales_df["seasonality"] = sales_df["month"].apply(seasonality)

# -------------------
# 9. ANOMALIES (REALISM 🔥)
# -------------------

# Returns (negative quantity)
return_indices = np.random.choice(sales_df.index, size=50, replace=False)
sales_df.loc[return_indices, "quantity"] *= -1

# High spikes
spike_indices = np.random.choice(sales_df.index, size=30, replace=False)
sales_df.loc[spike_indices, "quantity"] *= 5

# -------------------
# 10. REVENUE & PROFIT
# -------------------
sales_df["revenue"] = (
    sales_df["quantity"]
    * sales_df["adjusted_price"]
    * (1 - sales_df["discount"] / 100)
    * sales_df["seasonality"]
).round(2)

sales_df["profit"] = (
    sales_df["quantity"]
    * (sales_df["adjusted_price"] - sales_df["cost"])
).round(2)

# -------------------
# 11. CUSTOMER SEGMENTATION 🔥
# -------------------
# Define segmentation function FIRST
def segment(val):
    if val > 50000:
        return "VIP"
    elif val > 20000:
        return "Regular"
    else:
        return "Low Value"

# Aggregate revenue per customer
customer_revenue = sales_df.groupby("customer_id")["revenue"].sum().reset_index()

# Merge with customers (LEFT JOIN keeps all customers)
customers_df = customers_df.merge(customer_revenue, on="customer_id", how="left")

# Handle customers with no sales
customers_df["revenue"] = customers_df["revenue"].fillna(0)

# Apply segmentation AFTER fixing missing values
customers_df["segment"] = customers_df["revenue"].apply(segment)

# Add activity flag (VERY useful for analysis)
customers_df["has_sales"] = customers_df["revenue"] > 0

# -------------------
# 12. FINAL TABLES
# -------------------
sales_final = sales_df[[
    "sale_id", "date", "product_id", "customer_id",
    "employee_id", "quantity", "discount",
    "revenue", "profit"
]]

# -------------------
# 13. SAVE FILES
# -------------------
customers_df = customers_df.drop(columns=["revenue"])

customers_df.to_csv("customers.csv", index=False)
products_df.to_csv("products.csv", index=False)
employees_df.to_csv("employees.csv", index=False)
sales_final.to_csv("sales.csv", index=False)
regions_df.to_csv("regions.csv", index=False)

print("🚀 PRO DATASET GENERATED SUCCESSFULLY!")

🚀 PRO DATASET GENERATED SUCCESSFULLY!


In [14]:
import os
print(os.getcwd())

c:\Users\Lenovo\Documents\Enterprise_Sales_Workforce_Intelligence_System\scripts


In [15]:
import os

# -------------------
# CREATE PROJECT FOLDER IN DOCUMENTS
# -------------------

# Get user's Documents path
documents_path = os.path.expanduser("~/Documents")

project_name = "Enterprise_Sales_Workforce_Intelligence_System"
data_folder = os.path.join(documents_path, project_name, "data")

# Create folders if they don't exist
os.makedirs(data_folder, exist_ok=True)

# -------------------
# SAVE CSV FILES
# -------------------

customers_df.to_csv(os.path.join(data_folder, "customers.csv"), index=False)
products_df.to_csv(os.path.join(data_folder, "products.csv"), index=False)
employees_df.to_csv(os.path.join(data_folder, "employees.csv"), index=False)
sales_final.to_csv(os.path.join(data_folder, "sales.csv"), index=False)
regions_df.to_csv(os.path.join(data_folder,"regions.csv"), index=False)

print(f"✅ Files saved in: {data_folder}")

✅ Files saved in: C:\Users\Lenovo/Documents\Enterprise_Sales_Workforce_Intelligence_System\data
